<a href="https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ramyaa-hegde/flyrank-ml-internship-tasks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
* **Unit of Analysis:** One row = 1 unique content item (`content_hash_id`) aggregated by performance month (`month`).
* **Time Window:** Mid-panel evaluation month (`2026-03`). The final month (`2026-06`) is treated as a sealed evaluation set.

In [14]:
import os
import duckdb
from google.colab import userdata

# Load token from Secrets
hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()

# Set up official Hugging Face secret in DuckDB
con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}')")

# Query 1: Verify 1 row = 1 unique content item (using content_hash_id)
q1 = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT content_hash_id) as unique_content_items,
    MIN(month) as min_month,
    MAX(month) as max_month
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df_grain = con.execute(q1).df()
print("=== Section 1 Check: Grain & Window ===")
print(df_grain)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Section 1 Check: Grain & Window ===
   total_rows  unique_content_items min_month max_month
0     9841378                331437   2026-03   2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Context:** `content_hash_id`, `client_hash_id`, `month` (Identifies the content item, client, and observation window).
* **Label / Target:** `sessions_paid` (Primary downstream performance/conversion target metric).
* **Features:** `scroll_events` (Engagement signal available during observation period).
* **Excluded:** Future performance partitions or unaligned client metadata because including future time windows causes temporal data leakage.

In [11]:
# Query 2: Check features, context, and label metrics for March 2026
q2 = """
SELECT
    COUNT(*) as total_rows,
    COUNT(content_hash_id) as non_null_content_ids,
    AVG(scroll_events) as avg_scroll_events,
    AVG(sessions_paid) as avg_sessions_paid
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df_fields = con.execute(q2).df()
print("=== Section 2 Check: Fields & Counts ===")
print(df_fields)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Section 2 Check: Fields & Counts ===
   total_rows  non_null_content_ids  avg_scroll_events  avg_sessions_paid
0     9841378               9841378           0.032261           0.004418


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
# Query 3: Verification of grain, counts, and missing values
q3 = """
SELECT
    COUNT(*) as total_rows,
    COUNT(content_hash_id) as non_null_content_ids,
    COUNT(client_hash_id) as non_null_client_ids,
    SUM(CASE WHEN scroll_events IS NULL THEN 1 ELSE 0 END) as null_scroll_events,
    SUM(CASE WHEN sessions_paid IS NULL THEN 1 ELSE 0 END) as null_sessions_paid
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
"""

df_verify = con.execute(q3).df()
print("=== Section 3: Data Contract Verification ===")
print(df_verify)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Section 3: Data Contract Verification ===
   total_rows  non_null_content_ids  non_null_client_ids  null_scroll_events  \
0     9841378               9841378              9841378           3018741.0   

   null_sessions_paid  
0           3018741.0  


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

* **Unbalanced History:** Early rows in the warehouse may lack downstream conversion tracking (GSC-only early data).
* **Missing Off-Page Signals:** This data cannot capture external factors like off-site user intent, backlinks, or offline brand campaign impacts.
* **Aggregated Window Overlaps:** Monthly partition boundaries roll up daily events, masking intra-month intraday granularity.

In [13]:
# Query 4: Verify time window boundaries & null window checks
q4 = """
SELECT
    MIN(month) as earliest_month,
    MAX(month) as latest_month,
    COUNT(DISTINCT month) as total_months_available
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
"""

df_limits = con.execute(q4).df()
print("=== Section 4: Data Limits Check ===")
print(df_limits)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Section 4: Data Limits Check ===
  earliest_month latest_month  total_months_available
0        2025-01      2026-06                      18


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.